In [79]:
!pip install --upgrade pybaseball
!pip install MLB-StatsAPI

import pybaseball
from pybaseball import schedule_and_record
import ipywidgets as widgets
from IPython.display import display
import statsapi  # pip install MLB-StatsAPI
!pip install pybaseball
from pybaseball import statcast_pitcher
import requests
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
from datetime import datetime


In [8]:
!pip install pybaseball
from pybaseball import statcast_pitcher, statcast_batter, statcast, cache
import requests
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np
from datetime import datetime
from math import comb
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
cache.enable()

# =============================================================================
# CONSTANTS & LOOKUPS
# =============================================================================

LEAGUE_AVG_SPRINT_SPEED = 27.0
SPEED_BABIP_SCALAR      = 0.0015
SPEED_ADJ_CAP           = 0.012
H2H_CREDIBILITY_K       = 50    # ABs at which H2H gets 50% weight

teams = {
    "Arizona Diamondbacks": 109, "Atlanta Braves": 144, "Baltimore Orioles": 110,
    "Boston Red Sox": 111, "Chicago Cubs": 112, "Chicago White Sox": 145,
    "Cincinnati Reds": 113, "Cleveland Guardians": 114, "Colorado Rockies": 115,
    "Detroit Tigers": 116, "Houston Astros": 117, "Kansas City Royals": 118,
    "Los Angeles Angels": 108, "Los Angeles Dodgers": 119, "Miami Marlins": 146,
    "Milwaukee Brewers": 158, "Minnesota Twins": 142, "New York Mets": 121,
    "New York Yankees": 147, "Oakland Athletics": 133, "Philadelphia Phillies": 143,
    "Pittsburgh Pirates": 134, "San Diego Padres": 135, "San Francisco Giants": 137,
    "Seattle Mariners": 136, "St. Louis Cardinals": 138, "Tampa Bay Rays": 139,
    "Texas Rangers": 140, "Toronto Blue Jays": 141, "Washington Nationals": 120,
}

PITCH_NAMES = {
    'FF': 'Four-Seam Fastball', 'SI': 'Sinker',     'FC': 'Cutter',
    'SL': 'Slider',             'ST': 'Sweeper',     'CU': 'Curveball',
    'KC': 'Knuckle Curve',      'CH': 'Changeup',    'FS': 'Split-Finger',
    'KN': 'Knuckleball',        'SV': 'Slurve',      'SC': 'Screwball',
    'EP': 'Eephus',             'CS': 'Slow Curve',  'FA': 'Fastball',
}

PITCH_CONSOLIDATION = {
    'Curveball':     'Curveball',
    'Knuckle Curve': 'Curveball',
    'Slurve':        'Curveball',
    'Slow Curve':    'Curveball',
}

NON_PITCHES = ['IN', 'AB', 'PO']

# Strike zone grid: 3x3 buckets
# plate_x: negative = inside to RHB, positive = outside to RHB
ZONE_X_BINS = [-0.83, -0.28, 0.28, 0.83]
ZONE_Z_BINS = [1.5,    2.17,  2.83, 3.5 ]

ZONE_LABELS = {
    (0, 2): 'Up-In',   (1, 2): 'Up-Mid',   (2, 2): 'Up-Away',
    (0, 1): 'Mid-In',  (1, 1): 'Mid-Mid',  (2, 1): 'Mid-Away',
    (0, 0): 'Down-In', (1, 0): 'Down-Mid', (2, 0): 'Down-Away',
}


def consolidate_pitch(name):
    return PITCH_CONSOLIDATION.get(name, name)


# =============================================================================
# CACHES
# =============================================================================

_league_avg_cache      = {}
_league_zone_cache     = {}
_league_raw_df_cache   = None


# =============================================================================
# ZONE HELPERS
# =============================================================================

def assign_zone(px, pz):
    """Map pitch coordinates to (col, row) zone tuple, or None if out of range."""
    if px is None or pz is None:
        return None
    try:
        if np.isnan(px) or np.isnan(pz):
            return None
    except (TypeError, ValueError):
        return None
    col = next((i for i in range(3) if ZONE_X_BINS[i] <= px < ZONE_X_BINS[i + 1]), None)
    row = next((i for i in range(3) if ZONE_Z_BINS[i] <= pz < ZONE_Z_BINS[i + 1]), None)
    if col is None or row is None:
        return None
    return (col, row)


def get_pitcher_zone_profile_by_pitch(pitcher_df):
    """
    Returns {stand: {pitch_name: {zone: pct}}}
    What % of THIS pitch type does the pitcher throw to each zone.
    """
    result = {}
    for stand in ['R', 'L']:
        subset = pitcher_df[pitcher_df['stand'] == stand].copy()
        subset['zone'] = subset.apply(lambda r: assign_zone(r['plate_x'], r['plate_z']), axis=1)
        subset = subset[subset['zone'].notna()]
        result[stand] = {}
        for pitch, group in subset.groupby('pitch_name'):
            total  = len(group)
            counts = group['zone'].value_counts()
            result[stand][pitch] = {zone: round(cnt / total, 4) for zone, cnt in counts.items()}
    return result


def get_hitter_zone_profile_by_pitch(hitter_df):
    """
    Returns {p_throws: {pitch_name: {zone: xba}}}
    Hitter's xBA per zone, broken out by pitch type. Min 3 pitches per cell.
    """
    result = {}
    for hand in ['R', 'L']:
        subset = hitter_df[hitter_df['p_throws'] == hand].copy()
        subset['zone'] = subset.apply(lambda r: assign_zone(r['plate_x'], r['plate_z']), axis=1)
        subset = subset[subset['zone'].notna()]
        result[hand] = {}
        for pitch, group in subset.groupby('pitch_name'):
            result[hand][pitch] = {}
            for zone, zgroup in group.groupby('zone'):
                xba_vals = zgroup['estimated_ba_using_speedangle'].dropna()
                if len(xba_vals) >= 3:
                    result[hand][pitch][zone] = round(xba_vals.mean(), 3)
    return result


def get_league_zone_averages_by_pitch(league_df):
    """
    Returns {stand: {pitch_name: {zone: xba}}}
    League-avg xBA per zone per pitch type — baseline for normalization.
    """
    global _league_zone_cache
    if _league_zone_cache:
        return _league_zone_cache

    result = {}
    for stand in ['R', 'L']:
        subset = league_df[league_df['stand'] == stand].copy()
        subset['zone'] = subset.apply(lambda r: assign_zone(r['plate_x'], r['plate_z']), axis=1)
        subset = subset[subset['zone'].notna()]
        result[stand] = {}
        for pitch, group in subset.groupby('pitch_name'):
            result[stand][pitch] = {}
            for zone, zgroup in group.groupby('zone'):
                xba_vals = zgroup['estimated_ba_using_speedangle'].dropna()
                if not xba_vals.empty:
                    result[stand][pitch][zone] = round(xba_vals.mean(), 3)

    _league_zone_cache = result
    return result


def compute_zone_scalar_by_pitch(pitcher_zone_profile, hitter_zone_profile,
                                  league_zone_avgs, pitcher_pitch_summary,
                                  effective_bat_side, pitch_hand):
    """
    Computes a per-pitch zone scalar, blended by pitch usage.

    For each pitch:
        pitch_scalar = Σ_zones( pitcher_zone_pct × hitter_xba / league_xba )
    Final scalar = Σ_pitches( pitch_usage_pct × pitch_scalar )

    Returns (final_scalar, per_pitch_breakdown_dict).
    """
    pitcher_zones = pitcher_zone_profile.get(effective_bat_side, {})
    hitter_zones  = hitter_zone_profile.get(pitch_hand, {})
    league_zones  = league_zone_avgs.get(effective_bat_side, {})

    if not pitcher_zones or not hitter_zones or not league_zones or not pitcher_pitch_summary:
        return None, {}

    total_scalar = 0.0
    total_weight = 0.0
    per_pitch    = {}

    for pitch, usage_stats in pitcher_pitch_summary.items():
        usage_pct = usage_stats['usage'] / 100
        p_zones   = pitcher_zones.get(pitch, {})
        h_zones   = hitter_zones.get(pitch, {})
        lg_zones  = league_zones.get(pitch, {})

        if not p_zones or not h_zones or not lg_zones:
            per_pitch[pitch] = {
                'scalar': None, 'usage': usage_pct, 'covered_pct': 0,
                'reason': 'no zone data'
            }
            continue

        pitch_total = 0.0
        covered_pct = 0.0

        for zone, p_pct in p_zones.items():
            h_xba  = h_zones.get(zone)
            lg_xba = lg_zones.get(zone)
            if h_xba is None or lg_xba is None or lg_xba == 0:
                continue
            pitch_total += p_pct * (h_xba / lg_xba)
            covered_pct += p_pct

        if covered_pct < 0.25:
            per_pitch[pitch] = {
                'scalar': None, 'usage': usage_pct, 'covered_pct': covered_pct,
                'reason': f'low coverage ({covered_pct:.0%})'
            }
            continue

        pitch_scalar = round(pitch_total / covered_pct, 4)
        per_pitch[pitch] = {
            'scalar': pitch_scalar, 'usage': usage_pct,
            'covered_pct': covered_pct, 'reason': None
        }
        total_scalar += usage_pct * pitch_scalar
        total_weight += usage_pct

    if total_weight < 0.3:
        return None, per_pitch

    return round(total_scalar / total_weight, 4), per_pitch


# =============================================================================
# LEAGUE AVERAGES
# =============================================================================

def get_league_averages():
    global _league_avg_cache, _league_raw_df_cache

    if _league_avg_cache:
        return _league_avg_cache, _league_raw_df_cache

    current_year = datetime.now().year
    start_date   = f"{current_year}-03-01"
    end_date     = datetime.now().strftime("%Y-%m-%d")

    print("Fetching league-wide Statcast data for baselines... (one-time, may take 30-60s)")
    df = statcast(start_dt=start_date, end_dt=end_date)

    if df is None or df.empty:
        return {}, None

    df = df[df['pitch_type'].notna()]
    df = df[df['pitch_type'].str.strip() != '']
    df = df[~df['pitch_type'].isin(NON_PITCHES)]
    df['pitch_name'] = df['pitch_type'].map(lambda x: consolidate_pitch(PITCH_NAMES.get(x, x)))

    league_avgs = {}
    for stand in ['R', 'L']:
        subset = df[df['stand'] == stand]
        league_avgs[stand] = {}
        for pitch, group in subset.groupby('pitch_name'):
            xba_vals  = group['estimated_ba_using_speedangle'].dropna()
            xslg_vals = group['estimated_slg_using_speedangle'].dropna()
            league_avgs[stand][pitch] = {
                'xba':  round(xba_vals.mean(),  3) if not xba_vals.empty  else None,
                'xslg': round(xslg_vals.mean(), 3) if not xslg_vals.empty else None,
            }

    _league_avg_cache    = league_avgs
    _league_raw_df_cache = df
    return league_avgs, df


# =============================================================================
# ROSTER / PLAYER HELPERS
# =============================================================================

def get_pitchers(team_id):
    url      = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster?rosterType=active"
    response = requests.get(url)
    data     = response.json()
    return {
        p['person']['fullName']: p['person']['id']
        for p in data.get('roster', [])
        if p.get('position', {}).get('code') == '1'
    }


def get_hitters(team_id):
    url      = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster?rosterType=active"
    response = requests.get(url)
    data     = response.json()
    return {
        p['person']['fullName']: p['person']['id']
        for p in data.get('roster', [])
        if p.get('position', {}).get('code') != '1'
    }


def get_player_handedness(player_id):
    url      = f"https://statsapi.mlb.com/api/v1/people/{player_id}"
    response = requests.get(url)
    data     = response.json()
    people   = data.get('people', [])
    if not people:
        return None, None
    p          = people[0]
    bat_side   = p.get('batSide',  {}).get('code')
    pitch_hand = p.get('pitchHand', {}).get('code')
    return bat_side, pitch_hand


# =============================================================================
# STATCAST PITCH DATA
# =============================================================================

def get_pitch_data(player_id):
    current_year = datetime.now().year
    start_date   = f"{current_year}-03-01"
    end_date     = datetime.now().strftime("%Y-%m-%d")

    df = statcast_pitcher(start_date, end_date, player_id)
    if df is None or df.empty:
        return None, None, None

    df = df[df['pitch_type'].notna()]
    df = df[df['pitch_type'].str.strip() != '']
    df = df[~df['pitch_type'].isin(NON_PITCHES)]
    df['pitch_name'] = df['pitch_type'].map(lambda x: consolidate_pitch(PITCH_NAMES.get(x, x)))

    def summarize(subset):
        total  = len(subset)
        result = {}
        for pitch, group in subset.groupby('pitch_name'):
            usage     = round(len(group) / total * 100, 1)
            xba_vals  = group['estimated_ba_using_speedangle'].dropna()
            xslg_vals = group['estimated_slg_using_speedangle'].dropna()
            xba       = round(xba_vals.mean(),  3) if not xba_vals.empty  else None
            xslg      = round(xslg_vals.mean(), 3) if not xslg_vals.empty else None
            result[pitch] = {'usage': usage, 'xba': xba, 'xslg': xslg, 'n': len(group)}
        return dict(sorted(result.items(), key=lambda x: -x[1]['usage']))

    return summarize(df[df['stand'] == 'R']), summarize(df[df['stand'] == 'L']), df


def get_batter_pitch_data(player_id):
    current_year = datetime.now().year
    start_date   = f"{current_year}-03-01"
    end_date     = datetime.now().strftime("%Y-%m-%d")

    df = statcast_batter(start_date, end_date, player_id)
    if df is None or df.empty:
        return None, None, None

    df = df[df['pitch_type'].notna()]
    df = df[df['pitch_type'].str.strip() != '']
    df = df[~df['pitch_type'].isin(NON_PITCHES)]
    df['pitch_name'] = df['pitch_type'].map(lambda x: consolidate_pitch(PITCH_NAMES.get(x, x)))

    def summarize(subset):
        total  = len(subset)
        result = {}
        for pitch, group in subset.groupby('pitch_name'):
            usage     = round(len(group) / total * 100, 1)
            xba_vals  = group['estimated_ba_using_speedangle'].dropna()
            xslg_vals = group['estimated_slg_using_speedangle'].dropna()
            xba       = round(xba_vals.mean(),  3) if not xba_vals.empty  else None
            xslg      = round(xslg_vals.mean(), 3) if not xslg_vals.empty else None
            result[pitch] = {'usage': usage, 'xba': xba, 'xslg': xslg, 'n': len(group)}
        return dict(sorted(result.items(), key=lambda x: -x[1]['usage']))

    return summarize(df[df['p_throws'] == 'R']), summarize(df[df['p_throws'] == 'L']), df


# =============================================================================
# PITCHER K-RATE
# =============================================================================

def get_pitcher_k_rate(pitcher_id):
    current_year = datetime.now().year
    url      = (f"https://statsapi.mlb.com/api/v1/people/{pitcher_id}/stats"
                f"?stats=season&season={current_year}&group=pitching")
    response = requests.get(url)
    data     = response.json()
    splits   = data.get('stats', [{}])[0].get('splits', [])
    if not splits:
        return None
    s  = splits[0].get('stat', {})
    ks = s.get('strikeOuts', 0)
    bf = s.get('battersFaced', 0)
    return round(ks / bf, 4) if bf > 0 else None


def get_league_avg_k_rate():
    return 0.225


def get_pitcher_k_ratio(pitcher_id):
    pitcher_k = get_pitcher_k_rate(pitcher_id)
    league_k  = get_league_avg_k_rate()
    if pitcher_k is None:
        return None, league_k, None
    return pitcher_k, league_k, round(pitcher_k / league_k, 4)


# =============================================================================
# BATTER SEASON STATS
# =============================================================================

def get_batter_season_stats(player_id):
    current_year = datetime.now().year
    url      = (f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
                f"?stats=season&season={current_year}&group=hitting")
    response = requests.get(url)
    data     = response.json()
    splits   = data.get('stats', [{}])[0].get('splits', [])
    if not splits:
        return None, None, None
    s        = splits[0].get('stat', {})
    avg      = s.get('avg')
    slg      = s.get('slg')
    ab       = s.get('atBats', 0)
    games    = s.get('gamesPlayed', 1)
    ab_per_g = round(ab / games, 2) if games > 0 else None
    return avg, slg, ab_per_g


def get_batter_k_rate(player_id):
    current_year = datetime.now().year
    url      = (f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
                f"?stats=season&season={current_year}&group=hitting")
    response = requests.get(url)
    data     = response.json()
    splits   = data.get('stats', [{}])[0].get('splits', [])
    if not splits:
        return None, None, None
    s          = splits[0].get('stat', {})
    pa         = s.get('plateAppearances', 0)
    ab         = s.get('atBats', 0)
    strikeouts = s.get('strikeOuts', 0)
    walks      = s.get('baseOnBalls', 0)
    hbp        = s.get('hitByPitch', 0)
    if pa == 0 or ab == 0:
        return None, None, None
    return (round(1 - strikeouts / ab, 4),
            round(strikeouts / ab, 4),
            round((walks + hbp) / pa, 4))


def get_batter_hit_breakdown(player_id):
    current_year = datetime.now().year
    url      = (f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
                f"?stats=season&season={current_year}&group=hitting")
    response = requests.get(url)
    data     = response.json()
    splits   = data.get('stats', [{}])[0].get('splits', [])
    if not splits:
        return None
    s       = splits[0].get('stat', {})
    singles = s.get('hits', 0) - s.get('doubles', 0) - s.get('triples', 0) - s.get('homeRuns', 0)
    doubles = s.get('doubles', 0)
    triples = s.get('triples', 0)
    hrs     = s.get('homeRuns', 0)
    tb      = singles + doubles * 2 + triples * 3 + hrs * 4
    if tb == 0:
        return None
    return {
        'singles':   {'count': singles, 'tb': singles,     'pct': round(singles     / tb * 100, 1)},
        'doubles':   {'count': doubles, 'tb': doubles * 2, 'pct': round(doubles * 2 / tb * 100, 1)},
        'triples':   {'count': triples, 'tb': triples * 3, 'pct': round(triples * 3 / tb * 100, 1)},
        'home_runs': {'count': hrs,     'tb': hrs * 4,     'pct': round(hrs * 4     / tb * 100, 1)},
        'total_tb':  tb,
    }


# =============================================================================
# HEAD-TO-HEAD
# =============================================================================

def get_head_to_head_stats(batter_id, pitcher_id):
    """
    Career H2H stats pulled from Statcast (pybaseball), aggregated across all
    available seasons. Returns (BA, AB) where AB counts only actual at-bats
    (excludes walks, HBP, sac flies).
    """
    STATCAST_FIRST_YEAR = 2015
    current_year = datetime.now().year

    AB_EVENTS  = {'single', 'double', 'triple', 'home_run',
                  'strikeout', 'field_out', 'grounded_into_double_play',
                  'double_play', 'triple_play', 'fielders_choice',
                  'fielders_choice_out', 'force_out', 'other_out',
                  'strikeout_double_play', 'field_error'}
    HIT_EVENTS = {'single', 'double', 'triple', 'home_run'}

    total_ab = 0
    total_h  = 0

    for year in range(STATCAST_FIRST_YEAR, current_year + 1):
        try:
            df = statcast_batter(f"{year}-03-01", f"{year}-11-30", batter_id)
            if df is None or df.empty:
                continue
            # Filter to plate appearances against this specific pitcher
            matchup = df[df['pitcher'] == pitcher_id]
            if matchup.empty:
                continue
            # One row per pitch — keep only the final pitch of each PA
            pa_endings = matchup[matchup['events'].notna()]
            ab_pa  = pa_endings[pa_endings['events'].isin(AB_EVENTS)]
            hit_pa = pa_endings[pa_endings['events'].isin(HIT_EVENTS)]
            total_ab += len(ab_pa)
            total_h  += len(hit_pa)
        except Exception:
            continue

    if total_ab == 0:
        return None, 0

    return round(total_h / total_ab, 4), total_ab


def blend_matchup_ba(matchup_xba, h2h_ba, h2h_ab, credibility_k=H2H_CREDIBILITY_K):
    """
    Blend matchup xBA with head-to-head career BA using credibility weighting.
    Requires >5 career ABs. Max H2H weight capped at 20%.
    """
    if matchup_xba is None:
        return matchup_xba, 0, None
    if h2h_ba is None or h2h_ab == 0 or h2h_ab <= 5:   # ← require >5 ABs
        return matchup_xba, 0, None
    raw_weight = h2h_ab / (h2h_ab + credibility_k)
    h2h_weight = min(raw_weight, 0.20)                   # ← cap at 20%
    blended    = round((1 - h2h_weight) * matchup_xba + h2h_weight * h2h_ba, 4)
    return blended, h2h_ab, h2h_weight


# =============================================================================
# SPEED ADJUSTMENT
# =============================================================================

def get_sprint_speed(player_id):
    """Fetch sprint speed (ft/sec) from the MLB Stats API."""
    current_year = datetime.now().year
    url      = (f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
                f"?stats=sprintSpeed&season={current_year}&group=hitting")
    response = requests.get(url)
    data     = response.json()
    splits   = data.get('stats', [{}])[0].get('splits', [])
    if not splits:
        return None
    speed = splits[0].get('stat', {}).get('sprintSpeed')
    return float(speed) if speed is not None else None


def compute_speed_adjustment(sprint_speed,
                              league_avg=LEAGUE_AVG_SPRINT_SPEED,
                              scalar=SPEED_BABIP_SCALAR,
                              cap=SPEED_ADJ_CAP):
    """
    Additive BA delta based on sprint speed vs league average.
    Capped at ±SPEED_ADJ_CAP. Only applied to balls-in-play BA.
    """
    if sprint_speed is None:
        return 0.0, None
    raw_adj = (sprint_speed - league_avg) * scalar
    adj     = round(max(-cap, min(cap, raw_adj)), 4)
    if sprint_speed >= 28.5:   tier = "elite speed"
    elif sprint_speed >= 27.5: tier = "above average"
    elif sprint_speed >= 26.5: tier = "average"
    elif sprint_speed >= 25.5: tier = "below average"
    else:                      tier = "slow"
    return adj, f"{sprint_speed:.1f} ft/sec ({tier})"


# =============================================================================
# MATCHUP COMPUTE
# =============================================================================

def compute_matchup_ba(vs_pitcher_split, vs_hitter_split, league_data):
    if vs_pitcher_split is None or vs_hitter_split is None:
        return None
    total = 0.0
    pitches_used = 0
    for pitch, pitcher_stats in vs_pitcher_split.items():
        usage  = pitcher_stats['usage'] / 100
        p_xba  = pitcher_stats['xba']
        lg_xba = league_data.get(pitch, {}).get('xba')
        h_stat = vs_hitter_split.get(pitch)
        h_xba  = h_stat['xba'] if h_stat else None
        if p_xba is None or lg_xba is None or h_xba is None:
            continue
        total        += usage * (p_xba / lg_xba) * h_xba
        pitches_used += 1
    return round(total, 3) if pitches_used > 0 else None


def compute_matchup_slg(vs_pitcher_split, vs_hitter_split, league_data):
    if vs_pitcher_split is None or vs_hitter_split is None:
        return None
    total = 0.0
    pitches_used = 0
    for pitch, pitcher_stats in vs_pitcher_split.items():
        usage   = pitcher_stats['usage'] / 100
        p_xslg  = pitcher_stats['xslg']
        lg_xslg = league_data.get(pitch, {}).get('xslg')
        h_stat  = vs_hitter_split.get(pitch)
        h_xslg  = h_stat['xslg'] if h_stat else None
        if p_xslg is None or lg_xslg is None or h_xslg is None:
            continue
        total        += usage * (p_xslg / lg_xslg) * h_xslg
        pitches_used += 1
    return round(total, 3) if pitches_used > 0 else None


def compute_hit_probabilities(matchup_slg, hit_breakdown):
    if matchup_slg is None or hit_breakdown is None:
        return None
    tb_weights = {'singles': 1, 'doubles': 2, 'triples': 3, 'home_runs': 4}
    return {
        hit_type: round(matchup_slg * (hit_breakdown[hit_type]['pct'] / 100) / tb_val, 4)
        for hit_type, tb_val in tb_weights.items()
    }


def compute_hit_type_probabilities(matchup_ba, hit_breakdown):
    if matchup_ba is None or hit_breakdown is None:
        return None
    total_hits = sum(hit_breakdown[k]['count'] for k in ['singles', 'doubles', 'triples', 'home_runs'])
    if total_hits == 0:
        return None
    return {
        hit_type: round(matchup_ba * hit_breakdown[hit_type]['count'] / total_hits, 4)
        for hit_type in ['singles', 'doubles', 'triples', 'home_runs']
    }


# =============================================================================
# BETTING ODDS
# =============================================================================

def compute_betting_odds(matchup_ba, matchup_slg, season_avg_f, season_slg_f,
                         hit_breakdown, k_rate_per_ab, ab_per_game):
    """
    matchup xBA/xSLG are BIP metrics — K-rate applied here for matchup ABs only.
    Season AVG/SLG are observed stats with Ks already reflected.
    """
    results = {}

    def to_ml(prob):
        if prob is None or prob <= 0:  return 'N/A'
        if prob >= 1.0:                return '+∞'
        if prob > 0.5:  return f"-{round((prob / (1 - prob)) * 100)}"
        else:           return f"+{round(((1 - prob) / prob) * 100)}"

    total_abs   = ab_per_game if ab_per_game is not None else 3.5
    matchup_abs = min(2.5, total_abs)
    season_abs  = max(0.0, total_abs - matchup_abs)
    kr          = k_rate_per_ab if k_rate_per_ab is not None else 0.0

    m_ba  = matchup_ba  * (1 - kr) if matchup_ba  is not None else None
    m_slg = matchup_slg * (1 - kr) if matchup_slg is not None else None
    s_ba  = season_avg_f
    s_slg = season_slg_f

    # --- At least 1 hit ---
    if m_ba is not None and s_ba is not None:
        p_no_hit = ((1 - m_ba) ** matchup_abs) * ((1 - s_ba) ** season_abs)
        results['1_hit'] = {'prob': round(1 - p_no_hit, 4)}
    else:
        results['1_hit'] = {'prob': None}

    # --- At least 2 hits ---
    if m_ba is not None and s_ba is not None:
        eff_ba = (m_ba * matchup_abs + s_ba * season_abs) / total_abs
        p0     = (1 - eff_ba) ** total_abs
        p1     = total_abs * eff_ba * ((1 - eff_ba) ** (total_abs - 1))
        results['2_hits'] = {'prob': round(1 - p0 - p1, 4)}
        # store for 3-hit calc
        _p0, _p1, _eff_ba = p0, p1, eff_ba
    else:
        results['2_hits'] = {'prob': None}
        _p0 = _p1 = _eff_ba = None

    # --- At least 3 hits ---
    if _p0 is not None and total_abs >= 2:
        p2 = comb(int(total_abs), 2) * (_eff_ba ** 2) * ((1 - _eff_ba) ** (total_abs - 2))
        results['3_hits'] = {'prob': round(1 - _p0 - _p1 - p2, 4)}
    else:
        results['3_hits'] = {'prob': None}

    # --- At least 2 TB ---
    if (hit_breakdown and m_ba is not None and s_ba is not None
            and m_slg is not None and s_slg is not None):
        total_hits = sum(hit_breakdown[k]['count'] for k in ['singles', 'doubles', 'triples', 'home_runs'])
        if total_hits > 0:
            hr_tb_pct = hit_breakdown['home_runs']['pct'] / 100
            hr_share  = hit_breakdown['home_runs']['count'] / total_hits
            dbl_share = hit_breakdown['doubles']['count']   / total_hits
            trp_share = hit_breakdown['triples']['count']   / total_hits
            sng_share = hit_breakdown['singles']['count']   / total_hits

            def blend(ba_p, slg_p): return (ba_p * 2 + slg_p) / 3

            mm_sng = blend(m_ba * sng_share, m_slg * (hit_breakdown['singles']['pct'] / 100) / 1)
            mm_dbl = blend(m_ba * dbl_share, m_slg * (hit_breakdown['doubles']['pct'] / 100) / 2)
            mm_trp = blend(m_ba * trp_share, m_slg * (hit_breakdown['triples']['pct'] / 100) / 3)
            mm_hr  = blend(m_ba * hr_share,  m_slg * hr_tb_pct / 4)
            ss_sng = blend(s_ba * sng_share, s_slg * (hit_breakdown['singles']['pct'] / 100) / 1)
            ss_dbl = blend(s_ba * dbl_share, s_slg * (hit_breakdown['doubles']['pct'] / 100) / 2)
            ss_trp = blend(s_ba * trp_share, s_slg * (hit_breakdown['triples']['pct'] / 100) / 3)
            ss_hr  = blend(s_ba * hr_share,  s_slg * hr_tb_pct / 4)

            m_no_hit = 1 - mm_sng - mm_dbl - mm_trp - mm_hr
            s_no_hit = 1 - ss_sng - ss_dbl - ss_trp - ss_hr

            p_no_hit     = (m_no_hit ** matchup_abs) * (s_no_hit ** season_abs)
            p_one_single = (
                matchup_abs * mm_sng * (m_no_hit ** (matchup_abs - 1)) * (s_no_hit ** season_abs) +
                season_abs  * ss_sng * (m_no_hit ** matchup_abs)       * (s_no_hit ** (season_abs - 1))
            )
            results['2_tb'] = {'prob': round(1 - p_no_hit - p_one_single, 4)}
        else:
            results['2_tb'] = {'prob': None}
    else:
        results['2_tb'] = {'prob': None}

    # --- Home run ---
    hr_prob_ba = hr_prob_slg = None
    if hit_breakdown and m_ba is not None and s_ba is not None:
        total_hits = sum(hit_breakdown[k]['count'] for k in ['singles', 'doubles', 'triples', 'home_runs'])
        if total_hits > 0:
            hr_share   = hit_breakdown['home_runs']['count'] / total_hits
            p_no_hr    = ((1 - m_ba * hr_share) ** matchup_abs) * ((1 - s_ba * hr_share) ** season_abs)
            hr_prob_ba = round(1 - p_no_hr, 4)
    if hit_breakdown and m_slg is not None and s_slg is not None:
        hr_tb_pct    = hit_breakdown['home_runs']['pct'] / 100
        p_no_hr      = ((1 - m_slg * hr_tb_pct / 4) ** matchup_abs) * ((1 - s_slg * hr_tb_pct / 4) ** season_abs)
        hr_prob_slg  = round(1 - p_no_hr, 4)

    if hr_prob_ba is not None and hr_prob_slg is not None:
        hr_final = round((hr_prob_ba + hr_prob_slg) / 2, 4)
    else:
        hr_final = hr_prob_ba or hr_prob_slg

    results['hr'] = {'prob': hr_final}

    for key in results:
        p = results[key]['prob']
        results[key]['ml'] = to_ml(p)

    return results


# =============================================================================
# DISPLAY HELPERS
# =============================================================================

def fmt_stat(val):
    if val is None: return '  N/A '
    return f".{str(round(val * 1000)).zfill(3)}"


def fmt_ratio(pitcher_val, league_val, default_on_missing='   N/A'):
    if pitcher_val is None or league_val is None:
        return default_on_missing
    return f"{pitcher_val / league_val:.2f}x"


def print_pitcher_split(label, pitcher_data, league_data):
    print(f"\n📊 {label}:")
    print(f"  {'Pitch':<25} {'Usage%':>7}  {'xBA':>6}  {'xSLG':>6}  {'xBA/lg':>7}  {'xSLG/lg':>8}  {'#':>6}")
    print(f"  {'-'*72}")
    if pitcher_data:
        for pitch, stats in pitcher_data.items():
            lg = league_data.get(pitch, {}) if league_data else {}
            print(
                f"  {pitch:<25} {stats['usage']:>6.1f}%"
                f"  {fmt_stat(stats['xba']):>6}"
                f"  {fmt_stat(stats['xslg']):>6}"
                f"  {fmt_ratio(stats['xba'],  lg.get('xba'),  default_on_missing='  1.00x'):>7}"
                f"  {fmt_ratio(stats['xslg'], lg.get('xslg'), default_on_missing='  1.00x'):>8}"
                f"  {stats['n']:>6}"
            )
    else:
        print("  No data available.")


def print_batter_split(label, batter_data, league_data):
    print(f"\n📊 {label}:")
    print(f"  {'Pitch':<25} {'Usage%':>7}  {'xBA':>6}  {'xSLG':>6}  {'xBA/lg':>7}  {'xSLG/lg':>8}  {'#':>6}")
    print(f"  {'-'*72}")
    if batter_data:
        for pitch, stats in batter_data.items():
            lg = league_data.get(pitch, {}) if league_data else {}
            print(
                f"  {pitch:<25} {stats['usage']:>6.1f}%"
                f"  {fmt_stat(stats['xba']):>6}"
                f"  {fmt_stat(stats['xslg']):>6}"
                f"  {fmt_ratio(stats['xba'],  lg.get('xba'),  default_on_missing='   N/A'):>7}"
                f"  {fmt_ratio(stats['xslg'], lg.get('xslg'), default_on_missing='   N/A'):>8}"
                f"  {stats['n']:>6}"
            )
    else:
        print("  No data available.")


# =============================================================================
# WIDGET STATE
# =============================================================================

pitcher_id_map = {}
hitter_id_map  = {}

team1_dropdown   = widgets.Dropdown(options=list(teams.keys()), description='Team 1:')
pitcher_dropdown = widgets.Dropdown(options=[], description='Pitcher:')
team2_dropdown   = widgets.Dropdown(options=list(teams.keys()), description='Team 2:')
hitter_dropdown  = widgets.Dropdown(options=[], description='Hitter:')
output           = widgets.Output()


def update_pitchers(change):
    global pitcher_id_map
    try:
        pitcher_id_map           = get_pitchers(teams[team1_dropdown.value])
        pitcher_dropdown.options = sorted(pitcher_id_map.keys()) or ["No pitchers found"]
    except Exception as e:
        pitcher_dropdown.options = [f"Error: {str(e)}"]


def update_hitters(change):
    global hitter_id_map
    try:
        hitter_id_map           = get_hitters(teams[team2_dropdown.value])
        hitter_dropdown.options = sorted(hitter_id_map.keys()) or ["No hitters found"]
    except Exception as e:
        hitter_dropdown.options = [f"Error: {str(e)}"]


team1_dropdown.observe(update_pitchers, names='value')
team2_dropdown.observe(update_hitters,  names='value')

button = widgets.Button(description="Submit", button_style='primary')


# =============================================================================
# MAIN CLICK HANDLER
# =============================================================================

def on_click(b):
    with output:
        output.clear_output()

        pitcher_name = pitcher_dropdown.value
        pitcher_id   = pitcher_id_map.get(pitcher_name)
        hitter_name  = hitter_dropdown.value
        hitter_id    = hitter_id_map.get(hitter_name)

        if not pitcher_id:
            print("Could not find pitcher ID.")
            return

        # ── Pitcher Statcast ──────────────────────────────────────────────────
        print(f"Fetching Statcast data for {pitcher_name}...")
        vs_right, vs_left, pitcher_df = get_pitch_data(pitcher_id)

        league_avgs, league_df = get_league_averages()
        lg_right = league_avgs.get('R', {})
        lg_left  = league_avgs.get('L', {})

        if vs_right is None and vs_left is None:
            print("No Statcast data found for this pitcher this season.")
        else:
            print(f"\n{'='*75}")
            print(f"  {pitcher_name} — Pitch Mix + xBA / xSLG vs League Avg ({datetime.now().year})")
            print(f"{'='*75}")
            print_pitcher_split("vs Right-Handed Batters", vs_right, lg_right)
            print_pitcher_split("vs Left-Handed Batters",  vs_left,  lg_left)

        pitcher_k_rate, league_k_rate, pitcher_k_ratio = get_pitcher_k_ratio(pitcher_id)

        # ── Batter Statcast ───────────────────────────────────────────────────
        if not hitter_id:
            print("\nCould not find hitter ID.")
            return

        print(f"\nFetching Statcast data for {hitter_name}...")
        vs_rhp, vs_lhp, hitter_df = get_batter_pitch_data(hitter_id)

        if vs_rhp is None and vs_lhp is None:
            print("No Statcast data found for this hitter this season.")
        else:
            print(f"\n{'='*75}")
            print(f"  {hitter_name} — xBA / xSLG by Pitch vs League Avg ({datetime.now().year})")
            print(f"{'='*75}")
            print_batter_split("vs Right-Handed Pitchers", vs_rhp, lg_right)
            print_batter_split("vs Left-Handed Pitchers",  vs_lhp, lg_left)

        # ── Handedness ────────────────────────────────────────────────────────
        bat_side, _   = get_player_handedness(hitter_id)
        _, pitch_hand = get_player_handedness(pitcher_id)

        if bat_side == 'S':
            effective_bat_side = 'L' if pitch_hand == 'R' else 'R'
        else:
            effective_bat_side = bat_side

        pitcher_relevant_split = vs_right if effective_bat_side == 'R' else vs_left
        hitter_relevant_split  = vs_rhp   if pitch_hand == 'R'         else vs_lhp
        relevant_league        = lg_right if effective_bat_side == 'R' else lg_left

        # ── Base matchup xBA / xSLG ───────────────────────────────────────────
        matchup_ba  = compute_matchup_ba( pitcher_relevant_split, hitter_relevant_split, relevant_league)
        matchup_slg = compute_matchup_slg(pitcher_relevant_split, hitter_relevant_split, relevant_league)

        # ── Season stats ──────────────────────────────────────────────────────
        season_avg, season_slg, ab_per_game     = get_batter_season_stats(hitter_id)
        season_avg_f = float(season_avg) if season_avg else None
        season_slg_f = float(season_slg) if season_slg else None
        contact_rate_per_ab, k_rate_per_ab, walk_rate_per_pa = get_batter_k_rate(hitter_id)

        # ── Pitcher K-rate adjustment ─────────────────────────────────────────
        if k_rate_per_ab is not None and pitcher_k_ratio is not None:
            k_rate_adjusted = min(round(k_rate_per_ab * pitcher_k_ratio, 4), 0.99)
        else:
            k_rate_adjusted = k_rate_per_ab

        # ── Zone scalar (per-pitch) ───────────────────────────────────────────
        zone_scalar         = None
        per_pitch_breakdown = {}

        if pitcher_df is not None and hitter_df is not None and league_df is not None:
            pitcher_zone_profile = get_pitcher_zone_profile_by_pitch(pitcher_df)
            hitter_zone_profile  = get_hitter_zone_profile_by_pitch(hitter_df)
            league_zone_avgs     = get_league_zone_averages_by_pitch(league_df)

            zone_scalar, per_pitch_breakdown = compute_zone_scalar_by_pitch(
                pitcher_zone_profile, hitter_zone_profile, league_zone_avgs,
                pitcher_relevant_split, effective_bat_side, pitch_hand
            )

        # Apply zone scalar to base matchup xBA
        matchup_ba_zoned = (round(matchup_ba * zone_scalar, 4)
                            if matchup_ba is not None and zone_scalar is not None
                            else matchup_ba)

        # ── Head-to-head blend ────────────────────────────────────────────────
        h2h_ba, h2h_ab    = get_head_to_head_stats(hitter_id, pitcher_id)
        matchup_ba_blended, h2h_ab, h2h_weight = blend_matchup_ba(
            matchup_ba_zoned, h2h_ba, h2h_ab
        )

        # ── Speed adjustment ──────────────────────────────────────────────────
        sprint_speed              = get_sprint_speed(hitter_id)
        speed_adj, speed_context  = compute_speed_adjustment(sprint_speed)

        matchup_ba_final = (round(matchup_ba_blended + speed_adj, 4)
                            if matchup_ba_blended is not None else None)

        # Half-weight speed nudge to season avg (already partly reflects speed)
        if season_avg_f is not None:
            season_avg_f = round(season_avg_f + speed_adj * 0.5, 4)

        # ── Hit breakdown ─────────────────────────────────────────────────────
        hit_breakdown         = get_batter_hit_breakdown(hitter_id)
        hit_probs             = compute_hit_probabilities(matchup_slg, hit_breakdown)
        hit_type_probs        = compute_hit_type_probabilities(matchup_ba_final, hit_breakdown)
        season_hit_probs      = compute_hit_probabilities(season_slg_f, hit_breakdown)
        season_hit_type_probs = compute_hit_type_probabilities(season_avg_f, hit_breakdown)

        # ── Betting odds ──────────────────────────────────────────────────────
        betting_odds = compute_betting_odds(
            matchup_ba_final, matchup_slg, season_avg_f, season_slg_f,
            hit_breakdown, k_rate_adjusted, ab_per_game
        )

        # =====================================================================
        # OUTPUT
        # =====================================================================

        print(f"\n{'='*75}")
        print(f"  Matchup Projections: {pitcher_name} vs {hitter_name}")
        print(f"{'='*75}")
        print(f"  Pitcher throws: {pitch_hand}HP   |   Hitter bats: {bat_side} (effective: {effective_bat_side})")
        print(f"\n  {'':35} {'Matchup':>10}  {'Season':>10}")
        print(f"  {'-'*57}")
        print(f"  {'Base xBA (pitch mix only):':<35} {fmt_stat(matchup_ba):>10}")
        if zone_scalar is not None:
            print(f"  {'After zone scalar:':<35} {fmt_stat(matchup_ba_zoned):>10}  (×{zone_scalar:.3f})")
        if h2h_ba is not None and h2h_weight is not None:
            print(f"  {'After H2H blend:':<35} {fmt_stat(matchup_ba_blended):>10}  "
              f"(H2H: {fmt_stat(h2h_ba)}, {h2h_ab} ABs, {h2h_weight:.0%} wt)")
        else:
            print(f"  {'After H2H blend:':<35} {'N/A — 0 prior ABs':>10}")
        if speed_context:
            direction = f"+{speed_adj:.4f}" if speed_adj >= 0 else f"{speed_adj:.4f}"
            print(f"  {'After speed adj:':<35} {fmt_stat(matchup_ba_final):>10}  ({direction}, {speed_context})")
        print(f"  {'Slugging Percentage:':<35} {fmt_stat(matchup_slg):>10}  {season_slg if season_slg else 'N/A':>10}")
        print(f"  {'Season AVG (speed-nudged):':<35} {'':>10}  {fmt_stat(season_avg_f):>10}")
        print(f"  {'AB per Game:':<35} {'':>10}  {str(ab_per_game) if ab_per_game else 'N/A':>10}")

        # K-rate info
        if k_rate_per_ab is not None:
            print(f"\n  PA Rates — K% (per AB): {k_rate_per_ab:.1%}  |  "
                  f"BB/HBP% (per PA): {walk_rate_per_pa:.1%}  |  "
                  f"Contact%: {contact_rate_per_ab:.1%}")
            if pitcher_k_rate is not None:
                print(f"  Pitcher K%: {pitcher_k_rate:.1%} "
                      f"(vs league {league_k_rate:.1%} → {pitcher_k_ratio:.2f}x)  |  "
                      f"Adjusted hitter K%: {k_rate_adjusted:.1%}")

        # ── Zone scalar detail ─────────────────────────────────────────────────
        print(f"\n{'='*75}")
        print(f"  Zone Tendency Adjustment (per pitch)")
        print(f"{'='*75}")
        if zone_scalar is not None:
            direction = ("▲ favors hitter"  if zone_scalar > 1.02 else
                         "▼ favors pitcher" if zone_scalar < 0.98 else
                         "≈ neutral")
            print(f"  Composite zone scalar: {zone_scalar:.3f}x  ({direction})")
            print(f"\n  {'Pitch':<25} {'Usage%':>7}  {'Zone Scalar':>12}  {'Coverage':>9}")
            print(f"  {'-'*58}")
            for pitch, info in per_pitch_breakdown.items():
                scalar_str = (f"{info['scalar']:.3f}x" if info['scalar'] is not None
                              else f"N/A ({info.get('reason', '')})")
                print(f"  {pitch:<25} {info['usage']:>6.1%}  {scalar_str:>12}  {info['covered_pct']:>8.0%}")
        else:
            print(f"  Zone scalar: N/A (insufficient zone/pitch overlap data)")

        # ── xSLG hit probability breakdown ────────────────────────────────────
        if hit_probs and hit_breakdown:
            tb = hit_breakdown['total_tb']
            print(f"\n{'='*75}")
            print(f"  {hitter_name} — Hit Probabilities by Type · Matchup xSLG ({fmt_stat(matchup_slg)})")
            print(f"{'='*75}")
            print(f"  {'Hit Type':<15} {'Count':>6}  {'TB':>5}  {'% of TB':>8}  {'Prob':>7}  {'Bar'}")
            print(f"  {'-'*60}")
            for label, key in [('Singles','singles'),('Doubles','doubles'),('Triples','triples'),('Home Runs','home_runs')]:
                d   = hit_breakdown[key]
                bar = '█' * int(d['pct'] / 5)
                print(f"  {label:<15} {d['count']:>6}  {d['tb']:>5}  {d['pct']:>7.1f}%  {hit_probs[key]:>6.1%}  {bar}")
            print(f"  {'-'*60}")
            print(f"  {'Total TB':<15} {'':>6}  {tb:>5}")

        if season_hit_probs and hit_breakdown:
            tb = hit_breakdown['total_tb']
            print(f"\n{'='*75}")
            print(f"  {hitter_name} — Hit Probabilities by Type · Season SLG ({season_slg})")
            print(f"{'='*75}")
            print(f"  {'Hit Type':<15} {'Count':>6}  {'TB':>5}  {'% of TB':>8}  {'Prob':>7}  {'Bar'}")
            print(f"  {'-'*60}")
            for label, key in [('Singles','singles'),('Doubles','doubles'),('Triples','triples'),('Home Runs','home_runs')]:
                d   = hit_breakdown[key]
                bar = '█' * int(d['pct'] / 5)
                print(f"  {label:<15} {d['count']:>6}  {d['tb']:>5}  {d['pct']:>7.1f}%  {season_hit_probs[key]:>6.1%}  {bar}")
            print(f"  {'-'*60}")
            print(f"  {'Total TB':<15} {'':>6}  {tb:>5}")

        if hit_type_probs and hit_breakdown:
            total_hits = sum(hit_breakdown[k]['count'] for k in ['singles','doubles','triples','home_runs'])
            print(f"\n{'='*75}")
            print(f"  {hitter_name} — Hit Type Probabilities · Final Matchup BA ({fmt_stat(matchup_ba_final)})")
            print(f"{'='*75}")
            print(f"  {'Hit Type':<15} {'Count':>6}  {'% of Hits':>10}  {'Prob':>7}  {'Bar'}")
            print(f"  {'-'*55}")
            for label, key in [('Singles','singles'),('Doubles','doubles'),('Triples','triples'),('Home Runs','home_runs')]:
                d     = hit_breakdown[key]
                share = d['count'] / total_hits * 100
                bar   = '█' * int(share / 5)
                print(f"  {label:<15} {d['count']:>6}  {share:>9.1f}%  {hit_type_probs[key]:>6.1%}  {bar}")
            print(f"  {'-'*55}")
            print(f"  {'Total Hits':<15} {total_hits:>6}")

        if season_hit_type_probs and hit_breakdown:
            total_hits = sum(hit_breakdown[k]['count'] for k in ['singles','doubles','triples','home_runs'])
            print(f"\n{'='*75}")
            print(f"  {hitter_name} — Hit Type Probabilities · Season AVG ({season_avg})")
            print(f"{'='*75}")
            print(f"  {'Hit Type':<15} {'Count':>6}  {'% of Hits':>10}  {'Prob':>7}  {'Bar'}")
            print(f"  {'-'*55}")
            for label, key in [('Singles','singles'),('Doubles','doubles'),('Triples','triples'),('Home Runs','home_runs')]:
                d     = hit_breakdown[key]
                share = d['count'] / total_hits * 100
                bar   = '█' * int(share / 5)
                print(f"  {label:<15} {d['count']:>6}  {share:>9.1f}%  {season_hit_type_probs[key]:>6.1%}  {bar}")
            print(f"  {'-'*55}")
            print(f"  {'Total Hits':<15} {total_hits:>6}")

        # ── Betting odds ───────────────────────────────────────────────────────
        matchup_abs = min(2.5, ab_per_game) if ab_per_game else 2.5
        season_abs  = max(0.0, (ab_per_game or 3.5) - matchup_abs)
        print(f"\n{'='*75}")
        print(f"  {hitter_name} — Betting Odds")
        print(f"  ({ab_per_game or 3.5:.1f} ABs/game: {matchup_abs:.1f} matchup-weighted + {season_abs:.1f} season-weighted)")
        print(f"{'='*75}")
        print(f"  {'Prop':<25} {'Prob':>8}  {'ML Odds':>10}")
        print(f"  {'-'*47}")
        for label, key in [
            ('At Least 1 Hit',  '1_hit'),
            ('At Least 2 Hits', '2_hits'),
            ('At Least 3 Hits', '3_hits'),
            ('At Least 2 TB',   '2_tb'),
            ('Home Run',        'hr'),
        ]:
            r    = betting_odds[key]
            prob = f"{r['prob']:.1%}" if r['prob'] is not None else 'N/A'
            print(f"  {label:<25} {prob:>8}  {r['ml']:>10}")
        print(f"  {'-'*47}")

        # ── Footer ─────────────────────────────────────────────────────────────
        print(f"\n  Adjustment stack applied to matchup BA:")
        print(f"    1. Base xBA  →  pitch-mix weighted xBA vs league")
        if zone_scalar is not None:
            print(f"    2. Zone scalar ×{zone_scalar:.3f}  →  per-pitch hot/cold zone overlap")
        else:
            print(f"    2. Zone scalar  →  N/A (skipped)")
        if h2h_ba is not None and h2h_weight is not None:
            print(f"    3. H2H blend  →  {h2h_ab} ABs, {h2h_weight:.0%} weight on {fmt_stat(h2h_ba)} historical BA")
        else:
            print(f"    3. H2H blend  →  no prior matchup data")
        if speed_context:
            direction = f"+{speed_adj:.4f}" if speed_adj >= 0 else f"{speed_adj:.4f}"
            print(f"    4. Speed adj {direction}  →  {speed_context}")
        else:
            print(f"    4. Speed adj  →  N/A")
        if k_rate_per_ab is not None:
            print(f"    5. K-rate {k_rate_adjusted:.1%} applied to matchup ABs only (BIP correction)")
        print(f"\n  Season AVG/SLG used as-is (Ks already reflected). Speed nudge halved on season avg.")
        print(f"  Curve variants consolidated → Curveball. Min 3 pitches per zone/pitch cell.")
        print(f"\n  Team 1: {team1_dropdown.value}   |   Team 2: {team2_dropdown.value}")
        print(f"  * xBA/xSLG on balls in play only. League avg from all {datetime.now().year} Statcast data.")


button.on_click(on_click)

display(team1_dropdown, pitcher_dropdown, team2_dropdown, hitter_dropdown, button, output)
update_pitchers(None)
update_hitters(None)

Dropdown(description='Team 1:', options=('Arizona Diamondbacks', 'Atlanta Braves', 'Baltimore Orioles', 'Bosto…

Dropdown(description='Pitcher:', options=(), value=None)

Dropdown(description='Team 2:', options=('Arizona Diamondbacks', 'Atlanta Braves', 'Baltimore Orioles', 'Bosto…

Dropdown(description='Hitter:', options=(), value=None)

Button(button_style='primary', description='Submit', style=ButtonStyle())

Output()

List of fixes:
1. Print only prob of 1 hit, 2 hits, abs per game, and splits
2. Abs per game to only account for games where they start (games with 1 or less abs)
